In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

In [5]:

data_path = Path(r"C:\Users\Alfonso(FonsoP)Panli\Desktop\Projects\Disengagement\Data")

# Read Data
sc_engagement_raw = pd.read_excel(Path(data_path) / 'SC Engagement List2026-10-07 07_48_09.xlsx')
mystaff_raw = pd.read_excel(Path(data_path) / 'CONFIDENTIAL Mystaff 10052026.xlsx', sheet_name='MyStaff')


In [16]:

today = pd.Timestamp.today().normalize()

# Data Formatting
sc_engagement = sc_engagement_raw.copy()
sc_engagement['engagement_date'] = pd.to_datetime(sc_engagement['Engagement Date'])
sc_engagement['disengagement_date'] = pd.to_datetime(sc_engagement['Disengagement Date'], errors="coerce").fillna(today)

mystaff = mystaff_raw.copy()


In [79]:
# +1 when someone starts
starts = sc_engagement[["Shortcode", "engagement_date"]].rename(columns={"engagement_date": "date"}).copy()
starts["delta"] = 1

# -1 when someone ends (skip rows that were filled with today = still active)
ended = sc_engagement[sc_engagement["disengagement_date"] != today]
ends = ended[["Shortcode", "disengagement_date"]].rename(columns={"disengagement_date": "date"}).copy()
ends["delta"] = -1

events = pd.concat([starts, ends])

timeline = (
    events.groupby(["Shortcode", "date"], as_index=False)["delta"].sum()
          .sort_values(["Shortcode", "date"])
)
timeline["active_staff"] = timeline.groupby("Shortcode")["delta"].cumsum()
timeline = timeline.drop(columns="delta")

first = timeline.sort_values("date").groupby("Shortcode", as_index=False).first()

n_staff_start = 3
under_n = first[first["active_staff"] < n_staff_start]
under_n_short_code = under_n['Shortcode'].tolist()


tl = timeline[timeline["Shortcode"].isin(under_n_short_code)]

rows = []
for code, g in tl.sort_values("date").groupby("Shortcode"):
    start = g["date"].iloc[0]
    reached = g.loc[(g["date"] > start) & (g["active_staff"] >= 5), "date"]

    if len(reached):
        rows.append({
            "Shortcode": code,
            "start_date": start,
            "reached5_date": reached.iloc[0],
            "days_to_5": (reached.iloc[0] - start).days,
            "completed": True,
        })
    else:
        rows.append({
            "Shortcode": code,
            "start_date": start,
            "reached5_date": pd.NaT,
            "days_to_5": pd.NA,
            "completed": False,
        })

ramp = pd.DataFrame(rows)
ramp["days_to_5"] = ramp["days_to_5"].astype("Int64")   # whole numbers with NA allowed



In [80]:
ramp.sort_values('days_to_5').head(30)

,Shortcode,start_date,reached5_date,days_to_5,completed
564,CSChamps,2025-03-12,2025-03-13,1,True
1322,MYRDNLP,2019-09-02,2019-09-04,2,True
92,ARRIVIA,2021-06-21,2021-06-24,3,True
1379,OIL,2016-02-10,2016-02-15,5,True
1390,OOMA,2025-10-27,2025-11-03,7,True
681,DIAMONDBRACES,2022-11-07,2022-11-14,7,True
385,CIB106,2016-04-29,2016-05-06,7,True
2203,UCS,2021-09-06,2021-09-13,7,True
1148,LEWISELLIS,2021-05-10,2021-05-17,7,True
955,HIGHLAND,2017-10-23,2017-11-01,9,True


In [47]:
under_n.sort_values('active_staff')

,Shortcode,date,active_staff
474,CIB187,2018-03-20,-1
33,ADESSOCAPITAL,2025-09-22,0
2384,VERIFYDEBTSOLN,2023-11-20,0
2399,VIMWOOD,2024-11-11,0
2115,SEETORE,2023-10-31,0
...,...,...,...
1731,OPILINDUMPKNG,2024-11-11,2
2164,SNAPPY,2024-08-19,2
1878,POPUPTALENT,2021-02-16,2
2463,YOURINSURANCE,2020-02-24,2


In [66]:
timeline

,Shortcode,date,active_staff
0,05655543,2014-06-02,1
1,05655543,2014-08-14,2
2,05655543,2015-11-19,7
3,05655543,2016-01-04,8
4,05655543,2016-01-06,9
...,...,...,...
29765,ingenUIT,2024-02-26,1
29766,ingenUIT,2024-04-03,0
29767,opatdgenvi,2026-03-09,1
29768,opatdgenvi,2026-09-17,2


In [ ]:
# Data check

In [81]:
shortcode = 'JUNK2GO'

In [82]:
events[events['Shortcode']==shortcode].sort_values(by='date').head(14)

,Shortcode,date,delta
10345,JUNK2GO,2023-08-14,1
1258,JUNK2GO,2023-08-29,1
29565,JUNK2GO,2023-08-29,1
29124,JUNK2GO,2023-08-29,1
23024,JUNK2GO,2023-08-29,1
22338,JUNK2GO,2023-08-29,1
17793,JUNK2GO,2023-08-29,1
15845,JUNK2GO,2023-08-29,1
21446,JUNK2GO,2023-08-29,1
10762,JUNK2GO,2023-08-29,1


In [83]:
timeline[timeline['Shortcode']==shortcode].sort_values(by='date').head(14)

,Shortcode,date,active_staff
16134,JUNK2GO,2023-08-14,1
16135,JUNK2GO,2023-08-29,11
16136,JUNK2GO,2023-09-08,10
16137,JUNK2GO,2024-06-11,9
16138,JUNK2GO,2024-06-12,8
16139,JUNK2GO,2024-06-18,7
16140,JUNK2GO,2024-07-08,8
16141,JUNK2GO,2024-11-03,7
16142,JUNK2GO,2025-01-06,8
16143,JUNK2GO,2025-06-23,9


In [84]:
sc_engagement[sc_engagement['Shortcode']==shortcode].sort_values(by='engagement_date').head(14)

,Username,CSID,User Status,First Name,Middle Name,Last Name,Employment Status,Category,Engagement ID,Client Name,Shortcode,Engagement Status,Engagement Date,Disengagement Date,engagement_date,disengagement_date
10345,gracecar@cloudstaff.com,CS-11599,inactive,catherine grace,milla,cardinez,Resigned,Engaged,CS-11599-01,Junk2Go Limited,JUNK2GO,disengage,2023-08-14,2024-11-03,2023-08-14,2024-11-03
1258,AlyM@cloudstaff.com,CS-11447,inactive,alyssa marie,lising,mazon,Resigned,Engaged,CS-11447-01,Junk2Go Limited,JUNK2GO,disengage,2023-08-29,2026-03-25,2023-08-29,2026-03-25
23024,NelleS@cloudstaff.com,CS-11485,active,mary nelle,palma,saguit,Regular,Engaged,CS-11485-01,Junk2Go Limited,JUNK2GO,engage,2023-08-29,NaN,2023-08-29,2026-10-09
22338,milkhym@cloudstaff.com,CS-11490,inactive,milkhy,layug,martin,Resigned,Engaged,CS-11490-01,Junk2Go Limited,JUNK2GO,disengage,2023-08-29,2023-09-08,2023-08-29,2023-09-08
21446,MeiM@cloudstaff.com,CS-11482,inactive,marie may,musngi,musni,Resigned,Engaged,CS-11482-01,Junk2Go Limited,JUNK2GO,disengage,2023-08-29,2026-09-02,2023-08-29,2026-09-02
17793,KimMe@cloudstaff.com,CS-11481,inactive,kim,crusina,mendiola,Resigned,Engaged,CS-11481-01,Junk2Go Limited,JUNK2GO,disengage,2023-08-29,2026-04-03,2023-08-29,2026-04-03
29124,TiffanyPe@cloudstaff.com,CS-11486,active,tiffany,llanes,peñarroyo,Regular,Engaged,CS-11486-01,Junk2Go Limited,JUNK2GO,engage,2023-08-29,NaN,2023-08-29,2026-10-09
15845,joycant@cloudstaff.com,CS-11456,inactive,marijoy,lim,cantilleps,Resigned,Engaged,CS-11456-01,Junk2Go Limited,JUNK2GO,disengage,2023-08-29,2024-06-18,2023-08-29,2024-06-18
6101,CyMa@cloudstaff.com,CS-11457,active,jezcyrene,sese,mandap,Regular,Engaged,CS-11457-01,Junk2Go Limited,JUNK2GO,engage,2023-08-29,NaN,2023-08-29,2026-10-09
10762,heinzyg@cloudstaff.com,CS-11446,inactive,hainah,del rosario,yanga,Resigned,RTG,CS-11446-01,Junk2Go Limited,JUNK2GO,disengage,2023-08-29,2024-06-12,2023-08-29,2024-06-12


In [42]:
timeline.tail(1000)

,Shortcode,date,active_staff
28770,VIKINGCLOUD,2021-03-15,5
28771,VIKINGCLOUD,2021-03-17,6
28772,VIKINGCLOUD,2021-03-29,19
28773,VIKINGCLOUD,2021-04-05,25
28774,VIKINGCLOUD,2021-04-19,45
...,...,...,...
29765,ingenUIT,2024-02-26,1
29766,ingenUIT,2024-04-03,0
29767,opatdgenvi,2026-03-09,1
29768,opatdgenvi,2026-09-17,2
